<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    target_dir = f"{REPO_DIR}/work/notebooks"
    if os.path.basename(os.getcwd()) != "notebooks":
        os.chdir(target_dir)
print("Ready! Current Working Directory:", os.getcwd())

Ready! Current Working Directory: /content/flyrank-internship-assignment1/work/notebooks


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: "Our AI-driven content scores predict future traffic declines with 85% accuracy."

Methodology Question: Does the model rely on decision-derived features, such as product flags or existing-system scores, which would encode a circular result rather than learning the actual world?  
Finding 2: "Content that hasn't been updated in 12 months shows a massive drop in trailing 90-day traffic."

Methodology Question: Is there an overlapping window issue where the 90-day feature window contains the label's timeline, meaning the feature already knows the outcome at the time of prediction?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data & Prepare Target
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 2. Define Features (Excluding IDs and leaky label-derived columns)
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']
df = df.dropna(subset=features + ['is_declining_label', 'client_id'])

X = df[features]
y = df['is_declining_label']
groups = df['client_id']

# 3. Calculate Base Rate
base_rate = y.mean()

# 4. Train/Test - Naive Random Split (Before)
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.2, random_state=42)
rf_rnd = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_rnd.fit(X_train_rnd, y_train_rnd)

test_rnd_df = X_test_rnd.copy()
test_rnd_df['actual'] = y_test_rnd
test_rnd_df['prob'] = rf_rnd.predict_proba(X_test_rnd)[:, 1]
p50_rnd = test_rnd_df.sort_values('prob', ascending=False).head(50)['actual'].mean()

# 5. Train/Test - Honest Grouped Split by client_id (After)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)

test_grp_df = X_test_grp.copy()
test_grp_df['actual'] = y_test_grp
test_grp_df['prob'] = rf_grp.predict_proba(X_test_grp)[:, 1]
p50_grp = test_grp_df.sort_values('prob', ascending=False).head(50)['actual'].mean()

# 6. Report Findings
print(f"--- SPLIT COMPARISON ---")
print(f"Base Rate (Majority Class): {base_rate:.2f}")
print(f"Random Split Precision@50:  {p50_rnd:.2f} (Model memorizes client behavior)")
print(f"Grouped Split Precision@50: {p50_grp:.2f} (Honest performance on unseen clients)")

--- SPLIT COMPARISON ---
Base Rate (Majority Class): 0.57
Random Split Precision@50:  0.88 (Model memorizes client behavior)
Grouped Split Precision@50: 0.48 (Honest performance on unseen clients)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Group Leakage (The Primary Offender): Our initial random split allowed the model to memorize hidden client characteristics, falsely inflating our Precision@50 to 0.88. Switching to a grouped split by client_id exposed a massive 40-point gap, proving the model was relying on memorization rather than learning true predictive patterns for unseen data.  

Label-Derived Features: I successfully avoided the label trap by strictly excluding trend_pct and trend_direction, ensuring the label was not circularly computed from a feature column.  

Decision-Derived Features: No product flags or pre-existing system scores were used as inputs; they are correctly reserved strictly as baselines to beat, not as features.  

Overlapping Windows: All features (like impressions_90d) were strictly measured prior to the label's timeline to ensure the model couldn't look into the future.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original Bold Claim: "Our Random Forest model achieves 88% precision in predicting content decay, proving it is a highly accurate tool for prioritizing updates."

Rewritten Honest Claim: "We observed a 40-point performance gap when moving from a random split to an honest grouped split, indicating heavy client memorization. Our measured out-of-group Precision@50 (0.48) currently underperforms the naive base rate (0.57). While the features may provide weak directional hints, the model requires further refinement before it can be reliably deployed for decision-support